In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("pipeOptim.ipynb")

## Lecture Section

In this lecture, we will cover two important topics for building reliable machine learning workflows:

* Pipelines
* Cross Validation
* Optimization
    * Gradient descent
    * Common optimizers (SGD, Adam)
    * Learning rates and momentum
    * Hyperparameter tuning

## Pipelines

### Why Use Pipelines?

When we build a model, we often need to preprocess our data before fitting — for example, scaling features. A common mistake is to fit the scaler on the entire dataset, including the test set. This is called **data leakage**: information from the test set "leaks" into your preprocessing, making your model look better than it actually is on unseen data.

A `Pipeline` solves this by bundling preprocessing and modeling into a single object. When you call `.fit()` on a pipeline, it fits every step on the training data only. When you call `.predict()`, it transforms the new data using those already-fitted steps before passing it to the model.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Load a simple dataset
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

Without a pipeline, it's easy to accidentally leak data:

In [ ]:
# WRONG: scaler sees the entire dataset — this leaks test data!
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # leakage here
X_train_bad, X_test_bad = train_test_split(X_scaled, test_size=0.2, random_state=42)

In [ ]:
# CORRECT: use a Pipeline so the scaler is only fit on training data
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

### Accessing Pipeline Steps

You can inspect any step in a fitted pipeline using its name:

In [ ]:
# Access the fitted scaler
fitted_scaler = pipe.named_steps['scaler']
print("Feature means (from training data):", fitted_scaler.mean_[:5])

### Cross-Validation

When we evaluate a model, using a single train/test split can be unreliable. Depending on which examples happen to end up in the test set, our accuracy estimate might be different than what we would typically expect.

**K-fold cross-validation** is a more robust approach. The idea:

1. Split the training data into $k$ equal-sized chunks called **folds**.
2. Train the model on $k-1$ folds, evaluate on the remaining fold.
3. Repeat $k$ times, each time using a different fold as the validation set.
4. Average the $k$ scores.

With `cv=5`, we get 5 scores — one per fold — and typically report their mean and standard deviation.

A key point: cross-validation uses your **training data only**. The held-out test set is never touched during this process. CV is for model selection and evaluation during development; the test set is your final, one-time check.

### Pipelines with Cross-Validation

Pipelines work seamlessly with `cross_val_score`. Each fold correctly fits the preprocessing on the training portion only — no leakage across folds.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
print("CV Accuracy scores:", np.round(scores, 4))
print("Mean accuracy:", round(scores.mean(), 4))

## Optimization

### Gradient Descent

At the heart of training almost every ML model is **optimization**: finding the parameter values that minimize a loss function. **Gradient descent** is how this is commonly done using calculus principles. First you compute the gradient (slope) of the loss with respect to each parameter, then take a small step in the downhill direction. Then you repeat until you converge on a solution.

$$\theta \leftarrow \theta - \alpha \nabla_\theta L$$

where $\alpha$ is the **learning rate** — how big a step we take each iteration.

In [ ]:
import matplotlib.pyplot as plt

# Simple example: minimize f(x) = x^2 using gradient descent
# The gradient is f'(x) = 2x

def f(x):
    return x**2

def grad_f(x):
    return 2 * x

x = 8.0          # starting point
lr = 0.1         # learning rate
history = [x]

for _ in range(30):
    x = x - lr * grad_f(x)
    history.append(x)

xs = np.linspace(-9, 9, 200)
plt.figure(figsize=(8, 4))
plt.plot(xs, f(xs), label='f(x) = x²')
plt.scatter(history, [f(h) for h in history], color='red', zorder=5, s=30, label='GD steps')
plt.title('Gradient Descent on f(x) = x²')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Final x: {x:.6f}  (should be close to 0)")

### Learning Rate Matters

Too large a learning rate and the optimizer overshoots and diverges. Too small and it takes forever to converge. Let's visualize both.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
learning_rates = [0.01, 0.1, 0.95]
labels = ['Too small (lr=0.01)', 'Just right (lr=0.1)', 'Too large (lr=0.95)']

for ax, lr, label in zip(axes, learning_rates, labels):
    x = 8.0
    hist = [x]
    for _ in range(30):
        x = x - lr * grad_f(x)
        hist.append(x)
    ax.plot(xs, f(xs))
    ax.scatter(hist, [f(h) for h in hist], color='red', s=20, zorder=5)
    ax.set_title(label)
    ax.set_xlabel('x')
    ax.set_ylim(-1, 70)

axes[0].set_ylabel('f(x)')
plt.tight_layout()
plt.show()

### Common Optimizers: SGD and Adam

In practice, we don't compute the gradient over the full dataset every step (that's called **batch gradient descent**). Instead:

- **SGD (Stochastic Gradient Descent)**: computes the gradient on a single random example (or small mini-batch) at each step. Noisy but fast.
- **Adam**: keeps a running estimate of both the gradient and its squared magnitude, and adapts the learning rate for each parameter. Usually converges faster and requires less tuning.

In `sklearn`, you can choose the optimizer when using `SGDClassifier` or `SGDRegressor`. In Keras/PyTorch, you pass the optimizer as an argument to the compile step.


#### Momentum

The learning rate scales how big each individual step is and momentum scales how much of your previous velocity you carry forward. Momentum is like a learning rate that adapts based on history — if you've been consistently moving in the same direction across many steps, momentum causes you to accumulate speed in that direction. Your step size will be larger than your learning rate suggests. On the other hand, if the gradient keeps flipping direction (oscillating), the momentum terms cancel out and your effective step size shrinks (closer to your learning rate).

Generally, if you add momentum, you need to decrease your learning rate a bit. If you decrease momentum, you can increase your learning rate.

Think of it like a ball rolling downhill: it doesn't just respond to the slope at its current position, it also carries speed from rolling in that direction already. This helps the optimizer move faster through flat regions and dampen oscillations in narrow valleys.

 A value of 0 means no momentum (plain SGD); a value closer to 1 gives the optimizer a longer memory. Adam builds on this idea by tracking momentum for both the gradient and the squared gradient and that's why it's a good optimizer to use for a large set of problems.

Below we compare no momentum with momentum using a hand-made gradient.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def f(x):
    return x**4 - 3*x**2

def grad_f(x):
    return 4*x**3 - 6*x

n_steps = 50
x_start = 2.2
lr = 0.01

# Plain SGD
x = x_start
sgd_hist = [x]
for _ in range(n_steps):
    x = x - lr * grad_f(x)
    sgd_hist.append(x)

# SGD with momentum
x = x_start
velocity = 0.0
beta = 0.9
momentum_hist = [x]
for _ in range(n_steps):
    velocity = beta * velocity + (1 - beta) * grad_f(x)
    x = x - lr * velocity
    momentum_hist.append(x)

xs = np.linspace(-2.5, 2.5, 300)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, hist, title in zip(axes, [sgd_hist, momentum_hist], ['Plain SGD', 'SGD + Momentum (β=0.9)']):
    ax.plot(xs, f(xs), color='gray', linewidth=2, zorder=1)
    n = len(hist)
    colors = plt.cm.plasma(np.linspace(0, 1, n))
    for i in range(n):
        ax.scatter(hist[i], f(hist[i]), color=colors[i], s=40, zorder=5)
    ax.set_title(title)
    ax.set_xlabel('x')

axes[0].set_ylabel('f(x)')

# Colorbar to show time
sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(0, n_steps))
fig.colorbar(sm, ax=axes[1], label='Step')

plt.suptitle('SGD vs Momentum on f(x) = x⁴ - 3x²', y=1.02)
plt.tight_layout()
plt.show()

When we add momentum, we overshoot our target a bit, but we come back to it. The distance between our points, or the steps we take, also spread out when going downhill, and become closer together when nearing convergence.

### Hyperparameter Tuning with Grid Search

Choosing the right learning rate or regularization strength is an optimization problem. One approach is **grid search**: try combinations of hyperparameter values and pick the best one using cross-validation. It would be difficult to test every possible value combination, so we give a set of parameters we would like to try based on our prior knowledge of the problem. When we get our results, we can run a new grid search with a reduced, or more specific, set of parameters.


In [ ]:
from sklearn.model_selection import GridSearchCV

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

param_grid = {
    'model__C': [0.01, 0.1, 1.0, 10.0]  # C is the inverse of regularization strength
}

grid_search = GridSearchCV(pipe_lr, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print("Best C value:", grid_search.best_params_)
print("Best CV accuracy:", round(grid_search.best_score_, 4))
print("Test accuracy with best model:", round(accuracy_score(y_test, grid_search.predict(X_test)), 4))

## Assignment Section

<!-- BEGIN QUESTION -->

**Question 1.**

I tend to give my students a break around this time, and you are no exception! For 9 points, tell me your favorite thing you've learned this semester. Make sure to run the last question, too. it's a free point!

_Type your answer here, replacing this text._

<!-- END QUESTION -->

**Question 2.**

Make sure `x=3`!

In [ ]:
x = 3

In [ ]:
grader.check("q2")

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()